Görkem Kadir Solun

In [1]:
#@title Install Dependencies and Download Models

!uv pip install -q transformers datasets --prerelease disallow

import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import LogitsProcessor, set_seed
import numpy as np
import datasets

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [2]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model.to(device)
model = model.eval()

In [3]:
# @title Download lab files

import sys

!rm -rf llm_lab

![ ! -d 'llm_lab' ] && git clone https://github.com/ethz-privsec/llm_lab.git
%cd llm_lab
!git pull https://github.com/ethz-privsec/llm_lab.git
%cd ..
if "llm_lab" not in sys.path:
  sys.path.append("llm_lab")

Cloning into 'llm_lab'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 152 (delta 80), reused 114 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 503.66 KiB | 13.61 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/content/llm_lab
From https://github.com/ethz-privsec/llm_lab
 * branch            HEAD       -> FETCH_HEAD
Already up to date.
/content


In [4]:
# @title Example of how to generate 100 tokens of text without watermarking

from llm_lab.gpt_generate import generate_with_seed, gen_red_list

prompt = "Boston is one of the oldest municipalities in America,"
print(generate_with_seed(model, tokenizer, prompt, seed=42))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Boston is one of the oldest municipalities in America, but it's also among those with a history that dates back to at least 1857. The city has been home for most Aryan settlers from California until they settled here around 1787 when Columbus' New World arrived on Endurance Island and secured their land close by (the site was known as "Ferry Rock"). In other words: Kimball County doesn't exist right now — just be sure you're aware of its existence…
 [ Read more ] Thom Ann / Fox News $64 .99


You will now implement three different watermarking schemes:
1. A simple scheme that never outputs the letter 'e' (lowercase or uppercase)
2. A red-list scheme, that generates a random list of banned tokens for each token generation.
3. A soft red-list scheme, that also generates a random red-list, but just biases the LLM against these tokens instead of outright banning them, by substracting the value `logit_offset=2` from the logits of each red-listed token.

You should implement each of these schemes as a `LogitsProcessor` class.

For the red-list schemes, you should use `gpt_generate.gen_red_list` to generate a red list containing 50% of the LLM's vocabulary.
The seed for generating the pseudorandom red list is computed from the previous token processed by the model.

So for example, if the model has so far processed the string "my name is " (which tokenizes as `[1820, 1438, 318, 220]`), then the red list for the next token to be generated is `**gen_red_list(torch.LongTensor([220]), model.config.vocab_size)** = [43383,  7006, 40846, ...]`.

In [5]:
#@title Exercise 1: Implement a trivial watermarking scheme that samples text without any 'e' (lowercase or uppercase)

class NoEsLogitsProcessor(LogitsProcessor):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.e_token_ids = [
            token_id
            for token_id in range(tokenizer.vocab_size)
            if "e" in tokenizer.decode([token_id]).lower()
        ]

    def __call__(self, input_ids, scores):
        """
        Processes the output scores of the LLM before generating the next token.
        Args:
            input_ids: torch.LongTensor of shape (batch_size, sequence_length) — Indices of input sequence tokens in the vocabulary.
            scores: torch.FloatTensor of shape (batch_size, model.config.vocab_size) — Logits for the next token to be generated.
        Returns: torch.FloatTensor of shape (batch_size, model.config.vocab_size) — The processed logits.
        """
        scores[:, self.e_token_ids] = -float("inf")
        return scores


no_e_processor = NoEsLogitsProcessor()

prompt = "Anton Vowl is missing. Ransacking his Paris flat, a group of his faithful companions trawl through his diary for any hint as to his location and, insidiously, a ghost, from Vowl's past starts to cast its malignant shadow.\n "
output = generate_with_seed(model, tokenizer, prompt, logits_processor=no_e_processor, seed=42)
print(output)

Anton Vowl is missing. Ransacking his Paris flat, a group of his faithful companions trawl through his diary for any hint as to his location and, insidiously, a ghost, from Vowl's past starts to cast its malignant shadow.
  In fact that protagonist who was in such agony at first sight looks nothing but an ill-lucard son on all fours — sadistic bastard? A coward (and possibly also unkind) *cough* Mr N'Vow! Finally coming into contact with both Jonsonakos ("Christina Tyngrav") whom will do anything Ali affords him if it suits us; or two kilograms wolfish right off B1226 "Bitch" Gokai: about


In [6]:
#@title Exercise 2: Implement a red-list watermarking scheme

class RedListLogitsProcessor(LogitsProcessor):
    def __init__(self, red_frac=0.5, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.red_frac = red_frac

    def __call__(self, input_ids, scores):
        """
        Processes the output scores of the LLM before generating the next token.
        Args:
            input_ids: torch.LongTensor of shape (batch_size, sequence_length) — Indices of input sequence tokens in the vocabulary.
            scores: torch.FloatTensor of shape (batch_size, model.config.vocab_size) — Logits for the next token to be generated.
        Returns: torch.FloatTensor of shape (batch_size, model.config.vocab_size) — The processed logits.
        """
        vocab_size = scores.shape[-1]
        for batch_idx in range(input_ids.shape[0]):
            prev_token_id = int(input_ids[batch_idx, -1].item())
            red_list = gen_red_list(prev_token_id, vocab_size, frac_red=self.red_frac)
            red_list = torch.as_tensor(red_list, dtype=torch.long, device=scores.device)
            scores[batch_idx, red_list] = -float("inf")
        return scores

red_list_processor = RedListLogitsProcessor()

prompt = "Boston is one of the oldest municipalities in America,"
output = generate_with_seed(model, tokenizer, prompt, logits_processor=red_list_processor, seed=42)
print(output)

Boston is one of the oldest municipalities in America, but it's not a spiritual successor. It had been created after former Premier Harry Marnelli drew up legislation on behalf (in his first term as mayor) and appointed two new members: Daniel Blakeley Ryves from Endurance Township -- who served five terms earlier this year under current Mayor Philip Ayroyama — Michael Rieckenham Jr.. The township voted for Yang last week when an election was held against state ballot measures to dissolve public school systems because they have problems or defects with charter dogs


In [7]:
#@title Exercise 3: Implement a soft red-list watermarking scheme

class SoftRedListLogitsProcessor(LogitsProcessor):
    def __init__(self, red_frac=0.5, logit_offset=2.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.red_frac = red_frac
        self.logit_offset = logit_offset

    def __call__(self, input_ids, scores):
        """
        Processes the output scores of the LLM before generating the next token.
        Args:
            input_ids: torch.LongTensor of shape (batch_size, sequence_length) — Indices of input sequence tokens in the vocabulary.
            scores: torch.FloatTensor of shape (batch_size, model.config.vocab_size) — Logits for the next token to be generated.
        Returns: torch.FloatTensor of shape (batch_size, model.config.vocab_size) — The processed logits.
        """
        vocab_size = scores.shape[-1]
        for batch_idx in range(input_ids.shape[0]):
            prev_token_id = int(input_ids[batch_idx, -1].item())
            red_list = gen_red_list(prev_token_id, vocab_size, frac_red=self.red_frac)
            red_list = torch.as_tensor(red_list, dtype=torch.long, device=scores.device)
            scores[batch_idx, red_list] -= self.logit_offset
        return scores

soft_red_list_processor = SoftRedListLogitsProcessor()

prompt = "Boston is one of the oldest municipalities in America,"
output = generate_with_seed(model, tokenizer, prompt, logits_processor=soft_red_list_processor, seed=42)
print(output)

Boston is one of the oldest municipalities in America, but it's not a spiritual successor. It had been created after former Premier Harry Mudd and his family fled Sudan on Aug 13 2001 following atrocities by rebels there which left around 700 people dead including children under 18 years old who died at an open fire triggered earlier this year when Russian forces attacked rebel-controlled areas along military lines near Benghazi .
The elected members are from Sikh communities (with four more than LDS leaders) working alongside African Christians seeking local leadership to support both religious groups as well As for


Okay! We're now ready to start generating watermarked text.
We give you 20 prompts in `data/watermark_prompts.txt`.
For each of these, generate 100 more tokens using each of the three watermarking schemes, and save all of this as a numpy array for submission.

MAKE SURE TO USE `seed=42` FOR ALL YOUR GENERATIONS.

In [8]:
from tqdm import trange

with open('llm_lab/data/watermark_prompts.txt') as f:
  prompts = f.read().splitlines()

processors = [no_e_processor, red_list_processor, soft_red_list_processor]
outputs = []

seed = 42  # DON'T CHANGE THIS!!!

for i in trange(20):
  min_new_tokens = 100
  max_new_tokens = min_new_tokens

  for j in range(3):
    output = generate_with_seed(model, tokenizer, prompts[i], logits_processor=processors[j],
                                min_new_tokens=min_new_tokens, max_new_tokens=max_new_tokens, seed=seed)
    outputs.append(output)

print(outputs)

100%|██████████| 20/20 [01:20<00:00,  4.05s/it]

['The only way to deal with an unfree world is to become so absolutely free that your very existence is an act of rebellion. This was not a frivolous spiritual transformation, but had no such hard and fast solutions for human irrationality; it would simply start multiplying into farcical ways as you grow up in classically un-Christian schools…\nNot long ago I saw Martin Wolfram (who also holds Vatican authority) — who has actually said this kind words about Catholics: "I don\'t want anybody thinking or writing stuff." But if anything wolfram warns against Christianity! It\'s all too familiar looking at what Christians did', 'The only way to deal with an unfree world is to become so absolutely free that your very existence is an act of rebellion. It was no wonder then, in the Middle Ages it became common practice for landowners and princes (especially those who had a monopoly on land ownership or landed rents) mere sheep into cattle under different names: Jigdewenkubedt mit zoie van Leo

For the final part of this question, you now have to try and detect watermarked text.
We give you 80 pieces of text in `data/watermarked_gens.npy`.
For each piece of text you have to guess whether it was generated with:

1.   No watermark
2.   The dummy "no E's" watermark
3.   The red-list watermark
4.   The soft red-list watermark

We use the same `generate_with_seed` and `gen_red_list` implementations as you. Our red-list watermarking scheme also uses the same parameters (i.e., 50% of the tokens are red-listed, and for the soft version we substract 2.0 from the logits).

Exactly 20 of the 80 texts are generated with each of the 4 options above. Each text is comprised of a short prompt, followed by 100-200 generated tokens.

Store your guesses (1,2,3,4) for each piece of text in a numpy array.

In [9]:
from collections import defaultdict

outputs_secret = np.load("llm_lab/data/watermarked_gens.npy", allow_pickle=True)
assert len(outputs_secret) == 80

def red_list_rate(text, tokenizer, vocab_size, red_frac=0.5):
    token_ids = tokenizer.encode(str(text))
    if len(token_ids) < 2:
        return 0.0

    followers_by_prev = defaultdict(list)
    for prev_token, next_token in zip(token_ids[:-1], token_ids[1:]):
        followers_by_prev[int(prev_token)].append(int(next_token))

    red_hits = 0
    for prev_token, next_tokens in followers_by_prev.items():
        red_tokens = set(map(int, gen_red_list(prev_token, vocab_size, frac_red=red_frac)))
        red_hits += sum(next_token in red_tokens for next_token in next_tokens)

    return red_hits / (len(token_ids) - 1)

# The four classes are balanced. The dummy watermark is isolated by its tiny
# number of e/E characters, then the red-list schemes are ranked by red-hit rate.
e_counts = np.asarray([str(text).lower().count("e") for text in outputs_secret])
red_rates = np.asarray([
    red_list_rate(text, tokenizer, tokenizer.vocab_size)
    for text in outputs_secret
])

my_guesses = np.zeros(len(outputs_secret), dtype=int)

no_e_indices = np.argsort(e_counts)[:20]
my_guesses[no_e_indices] = 2

remaining = np.asarray([i for i in range(len(outputs_secret)) if my_guesses[i] == 0])
remaining_by_red_rate = remaining[np.argsort(red_rates[remaining])]

my_guesses[remaining_by_red_rate[:20]] = 3
my_guesses[remaining_by_red_rate[20:40]] = 4
my_guesses[remaining_by_red_rate[40:]] = 1

print({label: int(np.sum(my_guesses == label)) for label in [1, 2, 3, 4]})

{1: 20, 2: 20, 3: 20, 4: 20}


## Export your solution

To save your results, you can use the code below, which will save the file in Colab's temporary storage (or locally, if you're not using Colab), or on your Google Drive. If you save it on Colab's temporary storage, you can download it from there (see the file system icon on the left).

In [ ]:
from llm_lab.utils import get_solution_path, is_valid_student_id

#@markdown Check this box if you want to save your results on Google Drive. Otherwise they'll be
#@markdown saved on the ephimeral Colab storage. The storage will be deleted with the runtime,
#@markdown so REMEMBER TO DOWNLOAD THE FILES before you close the tab!
SAVE_ON_DRIVE = True # @param {"type":"boolean"}

#@markdown The number on your Legi (Student ID card). It's in the format 'dd-ddd-ddd'
STUDENT_ID = "25-936-154"  # @param {"type":"string","placeholder":"00-000-000"}

assert is_valid_student_id(STUDENT_ID), "Student ID should have the format 'dd-ddd-ddd'"

SOLUTIONS_PATH = get_solution_path(STUDENT_ID, SAVE_ON_DRIVE)

Mounted at /content/drive


In [11]:
# Save generations
assert len(outputs) == 60
np.save(SOLUTIONS_PATH / "Q1_gens.npy", np.asarray(outputs))

# Save guesses
assert len(my_guesses) == 80
np.save(SOLUTIONS_PATH / "Q1_guesses.npy", np.asarray(my_guesses))